# 02 — State-space structureStage 1, second half. Apply dimensionality reduction, then refuse to judge the result by eye.The deliverable of this notebook is a **table**: methods and preprocessing conditions down the rows, geometric descriptors across the columns. Not a gallery of embeddings.

In [ ]:
%load_ext autoreload%autoreload 2import numpy as npimport matplotlib.pyplot as pltimport pandas as pdfrom compbio2026 import data, geometry, plottingplotting.apply_style()rng = np.random.default_rng(2026)shd = data.load("train")english = shd.english_mask()          # 10 classes rather than 20 - cleaner story, faster fitsidx = np.flatnonzero(english)idx = rng.choice(idx, size=min(2000, len(idx)), replace=False)X, y, t = data.build_design_matrix(shd, bin_ms=10.0, t_max_ms=800.0, smooth_ms=20.0, trials=idx)Xf = data.flatten(X)print(Xf.shape)

## Start with PCA, alwaysFast, deterministic, and it tells you the effective dimensionality. If PCA already separates the classes, most of the nonlinear machinery is decoration.

In [ ]:
from sklearn.decomposition import PCApca = PCA(n_components=50).fit(Xf)Z_pca = pca.transform(Xf)fig, axes = plt.subplots(1, 2, figsize=(11, 4))axes[0].plot(np.cumsum(pca.explained_variance_ratio_), marker="o", ms=3, color=plotting.ACCENT)axes[0].axhline(0.9, ls="--", lw=1, color=plotting.INK_MUTED)axes[0].set_xlabel("Components")axes[0].set_ylabel("Cumulative variance explained")plotting.embedding(Z_pca, y, labels=data.DIGIT_KEYS, ax=axes[1])axes[1].legend(ncol=2, fontsize=7)axes[1].set_title("PCA")fig.tight_layout()print(f"participation ratio: {geometry.participation_ratio(Xf):.1f} of {Xf.shape[1]} features")

The participation ratio is the effective dimensionality: `(sum lambda)^2 / sum lambda^2` over the covariance eigenvalues. Unlike "number of PCs to reach 90 %" it needs no threshold.

## Compare methods on one axisTrustworthiness measures how well an embedding preserves local neighbourhoods. It lets you put PCA, UMAP, Isomap and t-SNE on the same scale instead of arguing about which plot looks nicer.**t-SNE and UMAP caveat:** distances *between* clusters in these embeddings are not meaningful. Never quantify class separation from them. Use them to look, and quantify somewhere else.

In [ ]:
from sklearn.manifold import Isomap, SpectralEmbedding, TSNE# Reduce with PCA first - standard practice, and it makes the nonlinear methods tractable.Z50 = PCA(n_components=50).fit_transform(Xf)methods = {    "PCA": PCA(n_components=2).fit_transform(Xf),    "Isomap": Isomap(n_components=2, n_neighbors=15).fit_transform(Z50),    "Spectral": SpectralEmbedding(n_components=2, n_neighbors=15, random_state=0).fit_transform(Z50),    "t-SNE": TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(Z50),}try:    import umap    methods["UMAP"] = umap.UMAP(n_components=2, n_neighbors=15, random_state=0).fit_transform(Z50)except ImportError:    print("umap-learn not installed; skipping")

In [ ]:
fig, axes = plt.subplots(1, len(methods), figsize=(4 * len(methods), 3.8))for ax, (name, Z) in zip(np.atleast_1d(axes), methods.items()):    plotting.embedding(Z, y, labels=data.DIGIT_KEYS, ax=ax, s=6)    ax.set_title(name)    ax.set_xticks([]); ax.set_yticks([])np.atleast_1d(axes)[-1].legend(ncol=2, fontsize=6, loc="best")fig.tight_layout()

In [ ]:
rows = []for name, Z in methods.items():    rows.append({        "method": name,        "trustworthiness": geometry.trustworthiness(Z50, Z, n_neighbors=15),        "class_separation": geometry.class_separation(Z, y),        "participation_ratio": geometry.participation_ratio(Z),    })table = pd.DataFrame(rows).set_index("method")table.round(3)

Read that table carefully. A method can have high trustworthiness (it preserved neighbourhoods) and low class separation (the neighbourhoods it preserved are not about the digit). Those are different claims and they come apart.

## The control you must runRun the same pipeline on shuffled labels. If the embedding still looks structured, the structure is not about the digits — it is about something else in the data, or about the method.

In [ ]:
y_shuf = rng.permutation(y)Z = methods["PCA"]fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))plotting.embedding(Z, y, labels=data.DIGIT_KEYS, ax=axes[0], s=6); axes[0].set_title("true labels")plotting.embedding(Z, y_shuf, labels=data.DIGIT_KEYS, ax=axes[1], s=6); axes[1].set_title("shuffled labels")for ax in axes: ax.set_xticks([]); ax.set_yticks([])fig.tight_layout()print(f"class separation, true     : {geometry.class_separation(Z, y):.3f}")print(f"class separation, shuffled : {geometry.class_separation(Z, y_shuf):.3f}")

## Geometry across preprocessing conditionsThis sweep is what notebook 03 will regress decoding accuracy against. Build it now.

In [ ]:
rows = []for bin_ms in [5.0, 10.0, 20.0, 50.0]:    for smooth_ms in [0.0, 10.0, 30.0]:        Xc, yc, _ = data.build_design_matrix(shd, bin_ms=bin_ms, t_max_ms=800.0,                                             smooth_ms=smooth_ms, trials=idx)        Xcf = data.flatten(Xc)        rows.append({"bin_ms": bin_ms, "smooth_ms": smooth_ms, **geometry.summarize(Xcf, yc)})sweep = pd.DataFrame(rows)sweep.to_csv("../data/geometry_sweep.csv", index=False)sweep.round(3)

---## Exercises1. **Does trustworthiness predict class separation?** Plot one against the other across your methods. If they are uncorrelated, say what that means.2. **Neighbourhood size.** Re-run Isomap and UMAP with `n_neighbors` in {5, 15, 50}. How much does the picture change? How much does the *number* change? Which would you report?3. **Trajectories, not points.** So far each trial is one point. Instead embed each *time bin* and draw the path a trial takes. Do trials of the same digit follow similar paths? This is closer to what the transient-dynamics literature actually claims.4. **Factor analysis.** FA separates shared from private variance, which PCA does not. Does it change the effective dimensionality estimate?